In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets scikit-learn ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


# 1. Загрузка данных

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(uciml_id=222):
    dataset = fetch_ucirepo(id=uciml_id)
    X = dataset.data.features
    y = dataset.data.targets
    df = pd.concat([X, y], axis=1)

    feature_names = X.columns.to_list()
    target_name = 'y'

    # Преобразование целевой переменной в бинарный формат
    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.5, seed=42):

    # Разделение на train/test в соотношении 8 к 2
    train_df, test_df = train_test_split(
          df,
          test_size=test_size,
          random_state=seed,
          stratify=df[target_name]
      )

    return train_df, test_df

df, feature_names, target_name = load_dataset()
train_df, test_df = split_dataset(df, target_name)

df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,0
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,0
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,0
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  int64 
dtypes: int64(8), object(9)
memory usage: 5.9+ MB


# 2. Вспомогательные функции

In [ ]:
# Преобразование строки DataFrame в текстовое описание

def row_to_text_template(row, feature_names, target_name, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]
        # Форматирования значений в зависимости от типа
        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts )

    if include_target:
        target_value = "yes" if row[target_name] == 1 else "no"
        text += f" -> subscription: {target_value}"

    return text

print(row_to_text_template(train_df.iloc[0], feature_names, target_name, 1))
train_df.head(1)

The value of age is 36. The category of job is technician. The category of marital is divorced. The category of education is secondary. The category of default is no. The value of balance is 861. The category of housing is no. The category of loan is no. The category of contact is telephone. The value of day_of_week is 29. The category of month is aug. The value of duration is 140. The value of campaign is 2. The value of pdays is -1. The value of previous is 0. The value of poutcome is nan. -> subscription: no


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,aug,140,2,-1,0,NaN,0


In [ ]:
def parse_prediction(response):
    response = response.lower().strip()

    # Удаление пунктуаций
    response = response.rstrip('.,!?')

    # Поиск "yes" или "no" в ответе
    if response == "yes" or response.startswith("yes"):
        return 1
    elif response == "no" or response.startswith("no"):
        return 0
    elif "yes" in response.split():
        return 1
    elif "no" in response.split():
        return 0
    else:
        # Если не можем распознать, возвращаем случайное
        print(f"Warning: Could not parse response: '{response}'")
        return random.randint(0, 1)



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec


# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}...")


Пример текста:
The value of age is 40. The category of job is blue-collar. The category of marital is married. The category of education is primary. The category of default is no. The value of balance is 640. The category of housing is yes. The category of loan is yes. The value of contact is nan. The value of day...


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}...")


Пример текста:
The value of age is 40. The category of job is blue-collar. The category of marital is married. The category of education is primary. The category of default is no. The value of balance is 640. The category of housing is yes. The category of loan is yes. The value of contact is nan. The value of day...


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 2.5 7B Instruct
model_name = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device # Explicitly force all model layers to the detected device (GPU if available)
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Использовано памяти: 15.24 GB


In [ ]:
def create_prompt(row, feature_names, target_name, few_shot_examples=None):

    system_prompt = (
        "You are a classifier. Predict whether a bank client will subscribe: "
        "'yes' if they will subscribe, or 'no' if they won't. "
        "Answer with only one word: 'yes' or 'no'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"Client information: "
            f"{row_to_text_template(row, feature_names, target_name)}\n"
            f"Will this client subscribe to a term deposit?"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name)
            ex_target = "yes" if ex[target_name] == 1 else "no"
            messages.append({"role": "user",      "content": f"Client information: {ex_text}\nWill this client subscribe?"})
            messages.append({"role": "assistant", "content": ex_target})

        client_text = row_to_text_template(row, feature_names, target_name)
        messages.append({"role": "user", "content": f"Client information: {client_text}\nWill this client subscribe?"})

    return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True # добавление role assistant
            )


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

def predict_single(prompt, max_new_tokens=10):
    # Получение предсказания для одного примера

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Декодирирование только новых токенов
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip().lower()


# Тестирование инференса
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name)
test_response = predict_single(test_prompt)
test_prediction = parse_prediction(test_response)

print(f"Ответ модели: '{test_response}'")
print(f"Распознанное предсказание: {test_prediction}")
print(f"Истинная метка: {test_df.iloc[0][target_name]}")

Ответ модели: 'no'
Распознанное предсказание: 0
Истинная метка: 0


In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, model, tokenizer, device, max_new_tokens=10):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Текст ответа
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Вероятность из logits
    first_token_logits = outputs.scores[0][0]

    yes_ids = tokenizer.encode("yes", add_special_tokens=False)
    no_ids  = tokenizer.encode("no",  add_special_tokens=False)
    # print(f"[DEBUG] yes token ids: {yes_ids}, no token ids: {no_ids}")

    yes_logit = first_token_logits[yes_ids[0]]
    no_logit = first_token_logits[no_ids[0]]

    probs = F.softmax(torch.stack([yes_logit, no_logit]), dim=0)
    prob_yes = probs[0].item()

    return response, prob_yes

test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name)
test_response, test_prob = predict_single_with_prob(test_prompt, model, tokenizer, device)
test_pred = parse_prediction(test_response)

print(f"Ответ модели: '{test_response}'")
print(f"Распознанное предсказание: {test_prediction}")
print(f"Истинная метка: {test_df.iloc[0][target_name]}")

Ответ модели: 'no'
Распознанное предсказание: 0
Истинная метка: 0


# 3.1 Zero-shot

In [ ]:
print("ZERO-SHOT КЛАССИФИКАЦИЯ")

seed = 42

# Запускаем zero-shot предсказания
y_true_zero = []
y_pred_zero = []
y_prob_zero = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name)
    response, prob = predict_single_with_prob(prompt, model, tokenizer, device)
    prediction = parse_prediction(response)

    y_true_zero.append(row[target_name])
    y_pred_zero.append(prediction)
    y_prob_zero.append(prob)

zero_shot_time = time.time() - start_time

print(f"zero_shot_time: {zero_shot_time}")
print(f"zero_shot_time / len(y_true_zero) {zero_shot_time / len(y_true_zero)}")

ZERO-SHOT КЛАССИФИКАЦИЯ


  0%|          | 0/9043 [00:00<?, ?it/s]

zero_shot_time: 642.8544766902924
zero_shot_time / len(y_true_zero) 0.07108862951346814


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

drive_output_dir = "/content/drive/MyDrive/base_qwen"

# Сохранение LoRA weights + tokenizer
model.save_pretrained(drive_output_dir)
tokenizer.save_pretrained(drive_output_dir)

print(f"Сохранено: {drive_output_dir}")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Сохранено: /content/drive/MyDrive/base_qwen


In [ ]:
drive_output_dir = "/content/drive/MyDrive/base_qwen"
tokenizer = AutoTokenizer.from_pretrained(drive_output_dir)

model = AutoModelForCausalLM.from_pretrained(
    drive_output_dir,
    dtype=torch.bfloat16,
    device_map=device,
)
model.eval()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [ ]:
# Вычисляем метрики
roc_zero, f1_zero, acc_zero, pr_zero, rec_zero = compute_metrics(np.array(y_true_zero), np.array(y_pred_zero), np.array(y_prob_zero))

print("Результаты zero-shot:")
print(f"ROC AUC: {roc_zero}")
print(f"F1 Score: {f1_zero}")
print(f"Accuracy: {acc_zero}")
print(f"Precision: {pr_zero}")
print(f"Recall: {rec_zero}")

Результаты zero-shot:
ROC AUC: 0.5805053307655066
F1 Score: 0.27313432835820894
Accuracy: 0.8922923808470641
Precision: 0.648936170212766
Recall: 0.17296786389413987


In [ ]:
zero_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_zero),
    np.array(y_pred_zero),
    np.array(y_prob_zero),
    n_iter=1000
)

print("\nРезультаты zero-shot (bootstrap метрики с доверительными интервалами):")
for key, value in zero_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты zero-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.5803±0.0103
  F1: 0.2732±0.0164
  Accuracy: 0.8922±0.0032
  Precision: 0.6502±0.0280
  Recall: 0.1731±0.0119


ROC-AUC (0.58): модель практически не превосходит случайный классификатор.

Accuracy (0.892): В датасете Bank 88% клиентов не подписываются (класс "no"), поэтому если модель будет предсказывать "no" для всех, она получит 88% accuracy.

Precision (0.65), Recall (0.173): модель обнаруживает лишь около 17.3% среди всех подписчиков. Когда она говорит "yes", в 65% случаев она права.

Время тестирования: ~ 10 мин.

# 3.2 Few-shot

In [ ]:
drive_output_dir = "/content/drive/MyDrive/base_qwen"
tokenizer = AutoTokenizer.from_pretrained(drive_output_dir)

model = AutoModelForCausalLM.from_pretrained(
    drive_output_dir,
    dtype=torch.bfloat16,
    device_map=device,
)
model.eval()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [39]:
# Выбираем 64 примера из train для контекста
n_examples = 64

seed = 42

# Балансированная выборка
n_per_class = n_examples // 2
pos_examples = train_df[train_df[target_name] == 1].sample(n=n_per_class, random_state=seed)
neg_examples = train_df[train_df[target_name] == 0].sample(n=n_per_class, random_state=seed)
few_shot_df = pd.concat([pos_examples, neg_examples]).sample(frac=1, random_state=seed)

print(f"\nВыбрано {len(few_shot_df)} примеров для контекста:")
print(f"  Положительных: {(few_shot_df[target_name] == 1).sum()}")
print(f"  Отрицательных: {(few_shot_df[target_name] == 0).sum()}")

few_shot_examples = [row for _, row in few_shot_df.iterrows()]

# Few-shot предсказания
y_true_few = []
y_pred_few = []
y_prob_few = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, few_shot_examples)
    response, prob = predict_single_with_prob(prompt, model, tokenizer, device)
    prediction = parse_prediction(response)

    y_true_few.append(row[target_name])
    y_pred_few.append(prediction)
    y_prob_few.append(prob)

few_shot_time = time.time() - start_time

print(f"few_shot_time: {few_shot_time}")
print(f"few_shot_time / len(y_true_few) {few_shot_time / len(y_true_few)}")


Выбрано 64 примеров для контекста:
  Положительных: 32
  Отрицательных: 32


  0%|          | 0/9043 [00:00<?, ?it/s]

few_shot_time: 7161.6392250061035
few_shot_time / len(y_true_few) 0.7919539118662063


In [40]:
roc_few, f1_few, acc_few, pr_few, rec_few = compute_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few)
)
print("Результаты few-shot:")
print(f"ROC AUC: {roc_few}")
print(f"F1 Score: {f1_few}")
print(f"Accuracy: {acc_few}")
print(f"Precision: {pr_few}")
print(f"Recall: {rec_few}")


Результаты few-shot:
ROC AUC: 0.6551953509238139
F1 Score: 0.24607094746295466
Accuracy: 0.8143315271480703
Precision: 0.23438836612489308
Recall: 0.2589792060491493


In [41]:
few_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few),
    n_iter=1000
)

print("\nРезультаты few-shot (bootstrap метрики с доверительными интервалами):")
for key, value in few_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты few-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.6552±0.0090
  F1: 0.2460±0.0123
  Accuracy: 0.8142±0.0042
  Precision: 0.2345±0.0126
  Recall: 0.2589±0.0136


ROC-AUC (0.66): улучшение на 13% по сравнению с zero-shot. Модель начала лучше различать классы.

Precision (0.234), Recall (0.258): теперь она находит 26% клиентов (было 17%), но из тех, кому она говорит "yes", только 23% действительно подпишутся.

Accuracy (0.81): снизился из-за баланисировки выборок.

ROC-AUC улучшился, но практическая применимость ухудшилась.

Время выполнения: ~ 2ч.

Использование оперативная памяти графического процесса: 35.2/80GB

Графический процессор A100